In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("/Users/wenbo.wu/Documents/Documents - Wenbo WU/Thesis/RTE_use_case/RTE_dashboard/RTE2027July/fused data.csv", encoding="utf-8-sig")

years = list(range(2010, 2025))
cr_cols = [f"Nb CR {y}" for y in years if f"Nb CR {y}" in df.columns]

for c in cr_cols + ["Masse rejetée à chaque CR (kg)"]:
    df[c] = pd.to_numeric(
        df[c].astype(str).str.replace(",", ".", regex=False),
        errors="coerce"
    )

threshold_cr = 365 / 60

def calc_equipment_leakage(row):
    total = 0.0
    stop_year = None

    for y in years:
        col = f"Nb CR {y}"
        if col not in row.index:
            continue

        nb_cr = row[col]
        mass = row["Masse rejetée à chaque CR (kg)"]

        if pd.isna(nb_cr) or pd.isna(mass):
            continue

        yearly_leak = nb_cr * mass
        total += yearly_leak

        if yearly_leak >= 5 or (365 / nb_cr) <= 60:
            stop_year = y
            break

    return pd.Series({
        "equipment_leakage": total,
        "stop_year": stop_year
    })

result = df.apply(calc_equipment_leakage, axis=1)

output = pd.concat([df[["N Equipement"]], result], axis=1)

total_leakage = output["equipment_leakage"].sum()

output.to_csv(
    "Result_simulationCriteria/simulation_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Total leakage:", total_leakage)
print("Number of CSEMs that triggered criteria:", output["stop_year"].notna().sum())

Total leakage: 17865.02665
Number of CSEMs that triggered criteria: 681


In [5]:
# more results print

import pandas as pd
import numpy as np
import os

df = pd.read_csv(
    "/Users/wenbo.wu/Documents/Documents - Wenbo WU/Thesis/RTE_use_case/RTE_dashboard/RTE2027July/fused data.csv",
    encoding="utf-8-sig"
)

C_LEAK = 2520.0
C_MANT = 47000
years = list(range(2010, 2025))
mass_col = "Masse rejetée à chaque CR (kg)"
cr_cols = [f"Nb CR {y}" for y in years if f"Nb CR {y}" in df.columns]

for c in cr_cols + [mass_col]:
    df[c] = pd.to_numeric(
        df[c].astype(str).str.replace(",", ".", regex=False),
        errors="coerce"
    )

threshold_cr = 365 / 60

def calc_equipment_leakage(row):
    total = 0.0
    stop_year = None
    repair_count = 0

    for y in years:
        col = f"Nb CR {y}"
        if col not in row.index:
            continue

        nb_cr = row[col]
        mass = row[mass_col]

        if pd.isna(nb_cr) or pd.isna(mass):
            continue

        yearly_leak = nb_cr * mass
        total += yearly_leak

        if yearly_leak >= 5 or (365 / nb_cr) <= 60:
            stop_year = y if stop_year is None else stop_year
            repair_count += 1
            break

    return pd.Series({
        "equipment_total_leak": total,
        "first_repair_year": stop_year,
        "repair_count": repair_count
    })

result = df.apply(calc_equipment_leakage, axis=1)

# =========================
# 1. 输出设备级结果
# =========================
result_df = df[["N Equipement"]].copy()
result_df["first_repair_year"] = result["first_repair_year"]
result_df["repair_count"] = result["repair_count"]
result_df["equipment_total_leak"] = result["equipment_total_leak"]
result_df["mass_per_cr_kg"] = df[mass_col]

for y in years:
    col = f"Nb CR {y}"
    if col in df.columns:
        result_df[col] = df[col]

# =========================
# 2. 输出年度汇总
# =========================
year_summary_records = []

for y in years:
    col = f"Nb CR {y}"

    if col not in df.columns:
        continue

    # 设备在该年份仍然处于计算范围内：
    # 1. 从未触发维修；或
    # 2. 首次维修年份 >= 当前年份
    eligible = (
        result_df["first_repair_year"].isna()
        | (result_df["first_repair_year"] >= y)
    )

    year_cr = df.loc[eligible, col]
    year_mass = df.loc[eligible, mass_col]

    yearly_leak = (year_cr * year_mass).sum(skipna=True)
    yearly_cr = year_cr.sum(skipna=True)

    year_summary_records.append({
        "year": y,
        "total_cr": yearly_cr,
        "total_leak_kg": yearly_leak,
        "leakage cost (€)": yearly_leak * C_LEAK,

        # 如果希望这里也保持原来的定义：统计非空 CR
        "active_equipment (noneZero CR)": year_cr.notna().sum(),

        "annual_repair_count": (
            result_df["first_repair_year"] == y
        ).sum(),

        "annual action cost (€)": (
            result_df["first_repair_year"] == y
        ).sum() * C_MANT
    })

year_summary_df = pd.DataFrame(year_summary_records)

# =========================
# 3. 总汇总
# =========================
summary_df = pd.DataFrame({
    "total_leak_kg": [result_df["equipment_total_leak"].sum()],
    "total_equipment": [len(result_df)],
    "repaired_equipment_count": [(result_df["repair_count"] > 0).sum()],
    "total_repair_actions": [result_df["repair_count"].sum()],
    "triggered_equipment_count": [result["first_repair_year"].notna().sum()]
})



# =========================
# consistency check
result_total = result_df["equipment_total_leak"].sum()
year_total = year_summary_df["total_leak_kg"].sum()

print("\n========== CONSISTENCY CHECK ==========")
print(f"Equipment total leak: {result_total:.6f} kg")
print(f"Year summary total leak: {year_total:.6f} kg")
print(f"Difference: {result_total - year_total:.12f} kg")
print(f"Consistent: {np.isclose(result_total, year_total)}")



# =========================
# 4. 保存文件
# =========================
os.makedirs("Result_simulationCriteria", exist_ok=True)

result_df.to_csv(
    "Result_simulationCriteria/simulation_results_equipment.csv",
    index=False,
    encoding="utf-8-sig"
)

year_summary_df.to_csv(
    "Result_simulationCriteria/simulation_results_year_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

summary_df.to_csv(
    "Result_simulationCriteria/simulation_results_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

# =========================
# 5. 控制台输出
# =========================
print("\n========== RESULT SUMMARY ==========")
print(summary_df)

print("\n========== YEAR SUMMARY ==========")
print(year_summary_df)

print("\n========== EQUIPMENT RESULT PREVIEW ==========")
print(result_df.head(10))



========== CONSISTENCY CHECK ==========
Equipment total leak: 17865.026650 kg
Year summary total leak: 17865.026650 kg
Difference: 0.000000000004 kg
Consistent: True

========== RESULT SUMMARY ==========
   total_leak_kg  total_equipment  repaired_equipment_count  \
0    17865.02665             2842                       681   

   total_repair_actions  triggered_equipment_count  
0                 681.0                        681  

========== YEAR SUMMARY ==========
    year  total_cr  total_leak_kg  leakage cost (€)  \
0   2010    1696.0    3332.860700      8.398809e+06   
1   2011    1076.0    1417.140562      3.571194e+06   
2   2012    1047.0    1170.351862      2.949287e+06   
3   2013    1119.0    1206.403563      3.040137e+06   
4   2014    1063.0    1142.652962      2.879485e+06   
5   2015    1134.0    1213.284463      3.057477e+06   
6   2016    1165.0    1231.747325      3.104003e+06   
7   2017    1006.0    1012.539025      2.551598e+06   
8   2018    1107.0    1290.7318